In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import copy
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras_tuner import RandomSearch, HyperModel
from keras_cv.losses import FocalLoss
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import tensorflow as tf

In [ ]:
df_org = pd.read_csv('/kaggle/input/emo-map-challenge/train_dataset.csv')
tdf_org = pd.read_csv('/kaggle/input/emo-map-challenge/test_dataset.csv')
df = copy.deepcopy(df_org)
tdf = copy.deepcopy(tdf_org)

In [ ]:
def pixel_array(x):
    return np.array(x.split(' ')).reshape(48, 48).astype('float32')

img_array = df['pixels'].apply(pixel_array)
img_array_test = tdf['pixels'].apply(pixel_array)

def convert_to_rgb(img_array):
    rgb_images = []
    for img in img_array:
        temp = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        rgb_images.append(temp)
    return np.array(rgb_images)

img_train = convert_to_rgb(img_array)
xtest = convert_to_rgb(img_array_test)

In [ ]:
img_labels = LabelEncoder().fit_transform(df['emotion'])
label_train = to_categorical(img_labels)

In [ ]:
emotion_text = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}
print(df.emotion.value_counts())
emotion_counts = df['emotion'].value_counts().to_dict()
emotions = [emotion_text[key] for key in emotion_counts.keys()]
counts = [emotion_counts[key] for key in emotion_counts.keys()]

plt.figure(figsize=(6, 6))
plt.bar(emotions, counts, color='skyblue')
plt.xlabel('Emotions')
plt.ylabel('Count')
plt.title('Emotion Distribution')
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))

k = 0
for label in sorted(df.emotion.unique()):
    selected_images = df[df.emotion == label].head(3)
    for index, row in selected_images.iterrows():
        # Convert the pixel string to a 48x48 image
        pic = np.array(row.pixels.split(' ')).reshape(48, 48).astype('float32')
        k += 1
        ax = plt.subplot(len(df.emotion.unique()), 3, k)
        ax.imshow(pic, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if k%3 == 1:
            ax.set_ylabel(emotion_text[label], fontsize=12, rotation=0, labelpad=50, va='center')

plt.tight_layout()
plt.show()

In [ ]:
happy_images = img_train[img_labels == 3]
happy_labels = img_labels[img_labels == 3]

other_classes_indices = img_labels != 3
other_images = img_train[other_classes_indices]
other_labels = img_labels[other_classes_indices]

smote = SMOTE(random_state=42)
other_images_reshaped = other_images.reshape((other_images.shape[0], -1))
other_images_resampled, other_labels_resampled = smote.fit_resample(other_images_reshaped, other_labels)

other_images_resampled = other_images_resampled.reshape((-1, 48, 48, 3))

img_train_resampled = np.concatenate((happy_images, other_images_resampled), axis=0)
label_train_resampled = np.concatenate((happy_labels, other_labels_resampled), axis=0)

label_train_resampled = to_categorical(label_train_resampled)

from collections import Counter
class_counts = Counter(np.argmax(label_train_resampled, axis=1))
for class_label, count in class_counts.items():
    print(f"Class {class_label} ({emotion_text[class_label]}): {count} images")


In [ ]:
class_weights = {0: 1.0042681395932713, 1: 4.025062656641603, 2: 0.9784735812133072, 3: 0.5668934240362812, 4: 0.8618832148243913, 5: 1.3382402141184342, 6: 0.907638315441783}

In [ ]:
xtrain, xval, ytrain, yval = train_test_split(img_train , label_train, test_size=0.08, random_state=42)
xtrain.shape, xval.shape, ytrain.shape, yval.shape

In [ ]:
convnext_base = tf.keras.applications.ConvNeXtBase(
    include_top=False,
    include_preprocessing=True,
    weights='imagenet',
    input_shape=(48, 48,3)
)

In [ ]:
efficientnet_base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(48, 48, 3)
    )

In [ ]:
def build_model_with_fixed_hyperparameters(base_model, num_classes):
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    
    # Fixed hyperparameters from tuning results
    units = 128
    dropout_rate = 0.4
    learning_rate = 1e-4
    alpha = 0.25
    gamma = 2.0
    
    x = Dense(units, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    
    focal_loss = FocalLoss(alpha=alpha, gamma=gamma, from_logits=False)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=focal_loss,
        metrics=['accuracy']
    )
    return model


In [ ]:
num_classes = ytrain.shape[1]
convnext_model = build_model_with_fixed_hyperparameters(convnext_base, num_classes)
efficientnet_model = build_model_with_fixed_hyperparameters(efficientnet_base, num_classes)

In [ ]:
batch_size = 32
epochs = 30
early_stopping = EarlyStopping(monitor='val_accuracy', min_delta = 0.0005, patience=11, verbose=1, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=7, min_lr=1e-7, verbose=1)
callbacks = [early_stopping, lr_scheduler]


In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True
)

In [ ]:
convnext_model.fit(
    train_datagen.flow(xtrain, ytrain, batch_size=batch_size),
    epochs=epochs,
    validation_data=(xval, yval),
    callbacks=callbacks
)

In [ ]:
efficientnet_model.fit(
    train_datagen.flow(xtrain, ytrain, batch_size=batch_size),
    epochs=epochs,
    validation_data=(xval, yval),
    callbacks=callbacks
)

In [ ]:
convnext_eval = convnext_model.evaluate(xval, yval)
efficientnet_eval = efficientnet_model.evaluate(xval, yval)

print("ConvNeXt Model Evaluation:", convnext_eval)
print("EfficientNet Model Evaluation:", efficientnet_eval)


In [ ]:
def ensemble_predictions(models, input_data):
    predictions = [model.predict(input_data) for model in models]
    return np.mean(predictions, axis=0)

ensemble_models = [convnext_model, efficientnet_model]
ensemble_preds = ensemble_predictions(ensemble_models, xval)

ensemble_preds_labels = np.argmax(ensemble_preds, axis=1)
yval_single_label = np.argmax(yval, axis=1)
print(classification_report(yval_single_label, ensemble_preds_labels))
print(confusion_matrix(yval_single_label, ensemble_preds_labels))


In [ ]:
ensemble_preds_test = ensemble_predictions(ensemble_models, xtest)
ensemble_preds_labels_test = np.argmax(ensemble_preds_test, axis=1)

In [ ]:
counts = np.bincount(ensemble_preds_labels_test, minlength=7)
for i in range(7):
    print(f"Number of {i}: {counts[i]}")

In [ ]:
submission = pd.DataFrame({
    'id': tdf['id'],
    'emotion': ensemble_preds_labels_test,
})

submission

In [ ]:
submission.to_csv('submission_conv_eff.csv',index=False)

In [ ]:
# focal_loss = FocalLoss(alpha=0.25, gamma=2, from_logits=False)
# # Compile and train EfficientNetV2B3 model
# efficientnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# efficientnet_history = efficientnet_model.fit(train_datagen.flow(xtrain, ytrain, batch_size=batch_size), epochs=epochs, validation_data=(xval, yval), callbacks=callbacks, class_weight=class_weights)

In [ ]:
# convnext_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# convnext_history = convnext_model.fit(train_datagen.flow(xtrain, ytrain, batch_size=batch_size), epochs=epochs, validation_data=(xval, yval), callbacks=callbacks, class_weight=class_weights)

In [ ]:
# val_predictions = ensemble_predict(efficientnet_model, convnext_model, xval)
# print(classification_report(np.argmax(yval, axis=1), val_predictions))
# print(confusion_matrix(np.argmax(yval, axis=1), val_predictions))